# 1. Business Understanding

The company is an online retailer that sells gift and home products. A lot of its customers look like wholesalers (they buy in big quantities), and most of the orders come from the UK.

**What the business wants to know**
- Which products and customers bring in the most money?
- When do people buy (month, weekday, hour)?
- How many orders get cancelled or returned?
- Are there wholesale type customers?
- Can we split customers into groups so marketing can treat each group differently?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)   # so the results are the same every time I run it

# 2. Data Audit

In [27]:
df = pd.read_excel('../data/Online Retail.xlsx')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [29]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
print('Cancelled lines (InvoiceNo starts with C):', df['InvoiceNo'].str.startswith('C').sum())
print('Lines with negative Quantity:', (df['Quantity'] < 0).sum())
print('Lines with UnitPrice = 0:', (df['UnitPrice'] == 0).sum())
print('Lines with UnitPrice < 0:', (df['UnitPrice'] < 0).sum())
print('Unique invoices:', df['InvoiceNo'].nunique())
print('Unique products:', df['StockCode'].nunique())
print('Unique customers:', df['CustomerID'].nunique())
print('Countries:', df['Country'].nunique())

Missing values per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows: 5268
Cancelled lines (InvoiceNo starts with C): 9288
Lines with negative Quantity: 10624
Lines with UnitPrice = 0: 2515
Lines with UnitPrice < 0: 2
Unique invoices: 25900
Unique products: 4070
Unique customers: 4372
Countries: 38


In [30]:
df[['Quantity', 'UnitPrice']].describe()

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


- Some rows have no CustomerID and some have no Description.
- There are duplicate rows.
- There are negative quantities (returns) and prices that are 0 or negative.
- Quantity and UnitPrice have very extreme min and max values, so there are outliers.

# 3. Cleaning & Data Quality Decisions

## 3.1 Data types and dates

In [31]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

In [32]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['StockCode'] = df['StockCode'].astype(str)
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce").astype("Int64")

print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [33]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              Int64
Country                object
dtype: object

## 3.2 Missing CustomerID

In [34]:
# first I group by invoice to see if a whole invoice is missing the ID or only some lines
inv_missing = df.groupby('InvoiceNo')['CustomerID'].apply(lambda x: x.isna().mean())

print('Invoices where ALL lines have no CustomerID:', (inv_missing == 1).sum())

missing = df[df['CustomerID'].isna()]
print('Share of rows with no CustomerID:', round(len(missing) / len(df) * 100, 1), '%')
print('Revenue of those rows:', round((missing['Quantity'] * missing['UnitPrice']).sum()))

Invoices where ALL lines have no CustomerID: 3710
Share of rows with no CustomerID: 24.9 %
Revenue of those rows: 1447682


**Decision:** The missing IDs belong to whole invoices (guest checkouts), not to random lines. So I do not drop them blindly.
- I keep them for product, time and country analysis because they are real sales.
- I exclude them from everything that needs a customer (customer ranking, RFM, clustering), because I cannot follow a guest from one order to the next.

**Business impact:** we lose some customers from the segmentation, but the sales totals stay correct.

In [35]:
df['IsGuest'] = df['CustomerID'].isna()

## 3.3 Duplicate rows

In [36]:
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Rows after removing exact duplicates:', len(df))

Duplicates before: 5268
Rows after removing exact duplicates: 536641


## 3.4 Cancellations and returns

In [37]:
df['IsCancel'] = df['InvoiceNo'].str.startswith('C')
cancel = df[df['IsCancel']]

print('Cancelled lines:', len(cancel))
print('Share of cancelled lines with negative Quantity:', round((cancel['Quantity'] < 0).mean() * 100, 1), '%')
print('Cancelled invoices:', cancel['InvoiceNo'].nunique())

Cancelled lines: 9251
Share of cancelled lines with negative Quantity: 100.0 %
Cancelled invoices: 3836


In [38]:
# how much money is lost because of cancellations?
gross_sales = (df.loc[~df['IsCancel'] & (df['Quantity'] > 0), 'Quantity'] *
               df.loc[~df['IsCancel'] & (df['Quantity'] > 0), 'UnitPrice']).sum()
returned_value = (cancel['Quantity'] * cancel['UnitPrice']).sum()    # negative number
net_revenue = gross_sales + returned_value

print('Gross sales   :', round(gross_sales))
print('Returns       :', round(returned_value))
print('Net revenue   :', round(net_revenue))

Gross sales   : 10619987
Returns       : -893980
Net revenue   : 9726007


**Decision:** I did not delete the negative quantities. They are returns, and they reduce the real revenue of the business.
- I keep sales and returns in two separate tables (made in 3.6), so I can analyse them separately.
- Net revenue = sales + returns (returns are negative numbers).
- For RFM, Frequency only counts real purchase invoices (a return is not a new purchase), but Monetary is the net amount.

**Business impact:** if I dropped the returns, revenue would look higher than it really is, and customers who return a lot would look better than they are.

## 3.5 Zero / negative prices and non product codes

In [39]:
bad_price = df[df['UnitPrice'] <= 0]
print('Lines with UnitPrice <= 0:', len(bad_price))
print('How many of them have no CustomerID:', bad_price['CustomerID'].isna().sum())
print()
print(bad_price['Description'].value_counts().head(10))

Lines with UnitPrice <= 0: 2512
How many of them have no CustomerID: 2472

Description
check                     159
?                          47
damages                    45
damaged                    43
found                      25
sold as set on dotcom      20
adjustment                 16
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64


In [40]:
# Some StockCodes are not products (POST = postage, M = manual, BANK CHARGES, DOT = dotcom postage...).
# They have no numbers in the code, so I use that to flag them.
df['IsProduct'] = df['StockCode'].str.contains(r'\d')

print('Non product lines:', (~df['IsProduct']).sum())
print(df.loc[~df['IsProduct'], 'StockCode'].value_counts().head(10))

Non product lines: 2790
StockCode
POST            1256
DOT              710
M                566
D                 77
S                 62
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
Name: count, dtype: int64


**Decision:** Lines with price 0 or negative are adjustments, damaged or lost stock, or giveaways. They are not real sales, so I isolate them in a separate table (`odd`). Lines like POST or BANK CHARGES are also not products, so they go to `odd` too. If I kept them they would show up as "top products".

## 3.6 Final tables

In [41]:
# real sales
sales = df[(~df['IsCancel']) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['IsProduct']].copy()

# real returns
returns = df[(df['IsCancel']) & (df['Quantity'] < 0) & (df['UnitPrice'] > 0) & df['IsProduct']].copy()

# everything else (adjustments, zero price, non products, negative quantity without a C)
odd = df[~df.index.isin(sales.index) & ~df.index.isin(returns.index)].copy()

print('sales  :', len(sales))
print('returns:', len(returns))
print('odd    :', len(odd))
print('check  :', len(sales) + len(returns) + len(odd) == len(df))

sales  : 522685
returns: 8670
odd    : 5286
check  : True


# 4. Feature Engineering

In [42]:
def add_features(d):
    d = d.copy()
    d['Year'] = d['InvoiceDate'].dt.year
    d['Month'] = d['InvoiceDate'].dt.month
    d['YearMonth'] = d['InvoiceDate'].dt.to_period('M').astype(str)
    d['DayOfWeek'] = d['InvoiceDate'].dt.day_name()
    d['Hour'] = d['InvoiceDate'].dt.hour
    d['TotalLineRevenue'] = d['Quantity'] * d['UnitPrice']
    d['Is_UK'] = (d['Country'] == 'United Kingdom').astype(int)
    return d

sales = add_features(sales)
returns = add_features(returns)
sales[['InvoiceNo', 'InvoiceDate', 'Year', 'Month', 'DayOfWeek', 'Hour', 'Quantity', 'UnitPrice', 'TotalLineRevenue', 'Is_UK']].head()

,InvoiceNo,InvoiceDate,Year,Month,DayOfWeek,Hour,Quantity,UnitPrice,TotalLineRevenue,Is_UK
0,536365,2010-12-01 08:26:00,2010,12,Wednesday,8,6,2.55,15.30,1
1,536365,2010-12-01 08:26:00,2010,12,Wednesday,8,6,3.39,20.34,1
2,536365,2010-12-01 08:26:00,2010,12,Wednesday,8,8,2.75,22.00,1
3,536365,2010-12-01 08:26:00,2010,12,Wednesday,8,6,3.39,20.34,1
4,536365,2010-12-01 08:26:00,2010,12,Wednesday,8,6,3.39,20.34,1


- `TotalLineRevenue` is the revenue of one **line** (not the whole order).
- `Is_UK` is 1 for United Kingdom and 0 for the rest, so I can compare UK with international sales.

# 5. Export Clean Dataset

In [43]:
# save the cleaned tables so the next notebooks can use them
df.to_csv('../data/clean_all.csv', index=False)
sales.to_csv('../data/sales.csv', index=False)
returns.to_csv('../data/returns.csv', index=False)
odd.to_csv('../data/odd.csv', index=False)
print('saved')

saved
